In [0]:
# Load Tables

from pyspark.sql.functions import col, to_date

fact_df = spark.table("medical_project.gold.fact_encounters")
date_df = spark.table("medical_project.gold.dim_date")

display(fact_df)


In [0]:
# Attach Date Dimension

fact_with_date = fact_df.withColumn(
    "encounter_date",
    to_date(col("start"))
).join(
    date_df,
    col("encounter_date") == col("date"),
    "left"
)

display(fact_with_date)

In [0]:
# Aggregate Counts

from pyspark.sql.functions import count

agg_df = fact_with_date.groupBy(
    "year", "quarter", "encounter_class"
).agg(
    count("encounter_id").alias("encounter_count")
)

display(agg_df)

In [0]:
# Total per Quarter

from pyspark.sql.window import Window
from pyspark.sql.functions import sum

window_spec = Window.partitionBy("year", "quarter")

agg_df = agg_df.withColumn(
    "total_encounters",
    sum("encounter_count").over(window_spec)
)

display(agg_df)

In [0]:
# Calculate Percentage

agg_df = agg_df.withColumn(
    "percentage",
    (col("encounter_count") / col("total_encounters")) * 100
)

display(agg_df)

In [0]:
# Final Output

kpi1 = agg_df.select(
    "year",
    "quarter",
    col("encounter_class").alias("encounter_class"),
    "encounter_count",
    "percentage"
).orderBy("year", "quarter")

display(kpi1)

In [0]:
# Save KPI Table

kpi1.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.kpi_encounter_mix")